# Week 1-3 · embedding 검색 계약과 threshold 이해하기

## 시나리오
외부 API나 DB 없이 작은 embedding adapter를 만들어 질문과 정책 문서의 유사도, top-k, relevance threshold를 관찰합니다.

## 학습 목표
- 텍스트를 vector로 바꾸는 embedding 계약을 구현한다.
- cosine similarity로 후보 순위를 계산한다.
- 임계값 아래 결과를 버려 unsupported 질문을 표현한다.

## 직접 조립
완성된 `weekX.app` 함수를 가져오지 않습니다. 아래 코드에서 작은 fixture와 핵심 객체·함수·연결을 직접 만듭니다.

### 1단계 · 작은 embedding adapter

In [ ]:
# 실행 순서: 1단계 · 작은 embedding adapter에서 MiniEmbedding, cosine_similarity, embed을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 1단계 · 작은 embedding adapter.
import math
import re
from collections import Counter
from langchain_core.documents import Document

# 외부 embedding API 대신 token 빈도로 검색의 vector 변환 역할만 재현합니다.
class MiniEmbedding:
    # 같은 token이 반복될수록 해당 차원의 빈도가 커지도록 Counter를 만듭니다.
    def embed(self, text: str) -> Counter:
        return Counter(re.findall(r"[가-힣A-Za-z0-9]+", text.lower()))

# 두 sparse vector의 방향 유사도를 계산하며 빈 vector는 0점으로 닫습니다.
def cosine_similarity(left: Counter, right: Counter) -> float:
    shared = set(left) & set(right)
    dot = sum(left[token] * right[token] for token in shared)
    left_norm = math.sqrt(sum(value * value for value in left.values()))
    right_norm = math.sqrt(sum(value * value for value in right.values()))
    return dot / (left_norm * right_norm) if left_norm and right_norm else 0.0

practice_corpus = [
    Document(page_content="휴가 신청은 시작일 3영업일 전에 제출합니다.", metadata={"chunk_id": "leave-01"}),
    Document(page_content="병가에는 진료 확인 자료를 첨부합니다.", metadata={"chunk_id": "sick-01"}),
]
mini_embedding = MiniEmbedding()

### 2단계 · 검색 함수 조립

In [ ]:
# 실행 순서: 2단계 · 검색 함수 조립에서 practice_search을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 2단계 · 검색 함수 조립.
# 질문을 vector화한 뒤 점수 정렬과 threshold를 순서대로 적용합니다.
def practice_search(query: str, relevance_threshold: float = 0.18, k: int = 2):
    query_vector = mini_embedding.embed(query)
    scored = [
        (cosine_similarity(query_vector, mini_embedding.embed(doc.page_content)), doc)
        for doc in practice_corpus
    ]
    ranked = sorted(scored, key=lambda item: item[0], reverse=True)[:k]
    return [(score, doc) for score, doc in ranked if score >= relevance_threshold]

supported = practice_search("휴가 신청은 언제 제출하나요?")
unsupported = practice_search("와이파이 비밀번호는 무엇인가요?")
[(round(score, 3), doc.metadata["chunk_id"]) for score, doc in supported]

### 3단계 · threshold 계약 확인

In [ ]:
# 실행 순서: 3단계 · threshold 계약 확인에서 fixture와 assertion을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 3단계 · threshold 계약 확인.
assert supported and supported[0][1].metadata["chunk_id"] == "leave-01"
assert unsupported == []
{"supported": len(supported), "unsupported": len(unsupported), "relevance_threshold": 0.18}

## 중간 결과
각 코드 셀의 출력에서 입력이 어떤 상태로 변했는지 확인합니다. 마지막 `assert`는 눈으로 본 결과를 실행 가능한 계약으로 고정합니다.

## 실패 경계
vector store가 top-k 문서를 돌려줬다는 이유만으로 근거가 있다고 판단하지 않습니다. 임계값 미달은 빈 목록으로 닫습니다.

## 실제 app 연결
Week 1 live app에서는 이 자리의 `MiniEmbedding`을 `OpenAIEmbeddings`로, 메모리 목록을 `PGVector`로 교체합니다. 그러나 `질문 embedding → 유사도 검색 → threshold → Document` 계약은 같습니다.

### 확장 과제
fixture의 문장이나 임계값을 하나 바꾸고, 어느 중간 결과와 assertion이 달라지는지 기록하세요.

## 다음 Notebook 연결
다음 `04_linear_stategraph.ipynb`에서는 검색과 답변 단계를 node로 만들고 선형 graph로 연결합니다.